In [45]:
# Import libraries
import pandas as pd
import sqlite3

In [46]:
# Load cleaned dataset
terror_df = pd.read_csv(
    "../data/processed/terrorism_cleaned.csv"
)
terror_df.head()

,eventid,iyear,imonth,iday,country_txt,region_txt,provstate,city,latitude,longitude,...,gname,weaptype1_txt,nkill,nwound,date,casualties,attack_success,suicide_attack,decade,high_casualty
0,197000000001,1970,7,2,Dominican Republic,Central America & Caribbean,National,Santo Domingo,18.456792,-69.951164,...,MANO-D,Unknown,1.0,0.0,1970-07-02,1.0,Successful,No,1970s,No
1,197000000002,1970,1,1,Mexico,North America,Federal,Mexico city,19.371887,-99.086624,...,23rd of September Communist League,Unknown,0.0,0.0,1970-01-01,0.0,Successful,No,1970s,No
2,197001000001,1970,1,1,Philippines,Southeast Asia,Tarlac,Unknown,15.478598,120.599741,...,Unknown Group,Unknown,1.0,0.0,1970-01-01,1.0,Successful,No,1970s,No
3,197001000002,1970,1,1,Greece,Western Europe,Attica,Athens,37.997490,23.762728,...,Unknown Group,Explosives,0.0,0.0,1970-01-01,0.0,Successful,No,1970s,No
4,197001000003,1970,1,1,Japan,East Asia,Fukouka,Fukouka,33.580412,130.396361,...,Unknown Group,Incendiary,0.0,0.0,1970-01-01,0.0,Successful,No,1970s,No


In [47]:
# Create SQLite database
conn = sqlite3.connect(
    "../data/processed/terrorism.db"
)
conn

In [48]:
# Write dataframe to database
terror_df.to_sql(
    "terrorism",
    conn,
    if_exists="replace",
    index=False
)

209706

In [49]:
# Verify tables
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

,name
0,terrorism


In [50]:
query = """
SELECT * FROM terrorism
LIMIT 5
"""

pd.read_sql(query, conn)

,eventid,iyear,imonth,iday,country_txt,region_txt,provstate,city,latitude,longitude,...,gname,weaptype1_txt,nkill,nwound,date,casualties,attack_success,suicide_attack,decade,high_casualty
0,197000000001,1970,7,2,Dominican Republic,Central America & Caribbean,National,Santo Domingo,18.456792,-69.951164,...,MANO-D,Unknown,1.0,0.0,1970-07-02,1.0,Successful,No,1970s,No
1,197000000002,1970,1,1,Mexico,North America,Federal,Mexico city,19.371887,-99.086624,...,23rd of September Communist League,Unknown,0.0,0.0,1970-01-01,0.0,Successful,No,1970s,No
2,197001000001,1970,1,1,Philippines,Southeast Asia,Tarlac,Unknown,15.478598,120.599741,...,Unknown Group,Unknown,1.0,0.0,1970-01-01,1.0,Successful,No,1970s,No
3,197001000002,1970,1,1,Greece,Western Europe,Attica,Athens,37.997490,23.762728,...,Unknown Group,Explosives,0.0,0.0,1970-01-01,0.0,Successful,No,1970s,No
4,197001000003,1970,1,1,Japan,East Asia,Fukouka,Fukouka,33.580412,130.396361,...,Unknown Group,Incendiary,0.0,0.0,1970-01-01,0.0,Successful,No,1970s,No


In [51]:
# Top 10 countries by terrorism incidents
query = """
SELECT
    country_txt,
    COUNT(*) AS total_incidents
FROM terrorism
GROUP BY country_txt
ORDER BY total_incidents DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,country_txt,total_incidents
0,Iraq,27521
1,Afghanistan,18920
2,Pakistan,15504
3,India,13929
4,Colombia,8915
5,Philippines,8271
6,Peru,6111
7,Yemen,6027
8,Nigeria,5550
9,United Kingdom,5513


In [52]:
# Countries with the highest fatalities
query = """
SELECT
    country_txt,
    SUM(nkill) AS total_fatalities
FROM terrorism
GROUP BY country_txt
ORDER BY total_fatalities DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,country_txt,total_fatalities
0,Iraq,81675.0
1,Afghanistan,67549.0
2,Nigeria,29093.0
3,Pakistan,25259.0
4,India,20358.0
5,Syria,18860.0
6,Sri Lanka,15588.0
7,Colombia,15135.0
8,Somalia,13175.0
9,Yemen,12842.0


In [53]:
# Distribution of attack types
query = """
SELECT
    attacktype1_txt,
    COUNT(*) AS incidents
FROM terrorism
GROUP BY attacktype1_txt
ORDER BY incidents DESC;
"""
pd.read_sql(query, conn)

,attacktype1_txt,incidents
0,Bombing/Explosion,98158
1,Armed Assault,49553
2,Assassination,21539
3,Hostage Taking (Kidnapping),14045
4,Facility/Infrastructure Attack,12325
5,Unknown,10942
6,Unarmed Assault,1229
7,Hostage Taking (Barricade Incident),1156
8,Hijacking,759


In [63]:
# Distribution of weapon types
query = """
SELECT
    weaptype1_txt,
    COUNT(*) AS incidents
FROM terrorism
WHERE weaptype1_txt <> 'Unknown'
GROUP BY weaptype1_txt
ORDER BY incidents DESC;
"""
pd.read_sql(query, conn)

,weaptype1_txt,incidents
0,Explosives,103475
1,Firearms,67648
2,Incendiary,13133
3,Melee,4307
4,Chemical,347
5,Sabotage Equipment,187
6,Vehicle (not to include vehicle-borne explosiv...,186
7,Other,136
8,Biological,36
9,Fake Weapons,35


In [64]:
# Most frequently attacked targets
query = """
SELECT
    targtype1_txt,
    COUNT(*) AS incidents
FROM terrorism
WHERE targtype1_txt <> 'Unknown'
GROUP BY targtype1_txt
ORDER BY incidents DESC;
"""
pd.read_sql(query, conn)

,targtype1_txt,incidents
0,Private Citizens & Property,51985
1,Military,34131
2,Police,28568
3,Government (General),23828
4,Business,22169
5,Transportation,7173
6,Utilities,6328
7,Religious Figures/Institutions,5107
8,Educational Institution,4761
9,Government (Diplomatic),3802


In [56]:
# Most active terrorist organizations
query = """
SELECT
    gname,
    COUNT(*) AS incidents
FROM terrorism
WHERE gname <> 'Unknown Group'
GROUP BY gname
ORDER BY incidents DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,gname,incidents
0,Taliban,11982
1,Islamic State of Iraq and the Levant (ISIL),7254
2,Shining Path (SL),4564
3,Al-Shabaab,4419
4,New People's Army (NPA),3395
5,Farabundo Marti National Liberation Front (FMLN),3351
6,Boko Haram,3320
7,Houthi extremists (Ansar Allah),3196
8,Irish Republican Army (IRA),2670
9,Kurdistan Workers' Party (PKK),2582


In [57]:
# Groups causing the most fatalities
query = """
SELECT
    gname,
    SUM(nkill) AS fatalities
FROM terrorism
WHERE gname <> 'Unknown Group'
GROUP BY gname
ORDER BY fatalities DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,gname,fatalities
0,Taliban,54118.0
1,Islamic State of Iraq and the Levant (ISIL),43352.0
2,Boko Haram,25528.0
3,Al-Shabaab,12277.0
4,Shining Path (SL),11608.0
5,Liberation Tigers of Tamil Eelam (LTTE),10778.0
6,Farabundo Marti National Liberation Front (FMLN),8065.0
7,Nicaraguan Democratic Force (FDN),6662.0
8,Houthi extremists (Ansar Allah),6420.0
9,Tehrik-i-Taliban Pakistan (TTP),6344.0


In [58]:
# Fatalities by region
query = """
SELECT
    region_txt,
    SUM(nkill) AS fatalities
FROM terrorism
GROUP BY region_txt
ORDER BY fatalities DESC;
"""
pd.read_sql(query, conn)

,region_txt,fatalities
0,Middle East & North Africa,149739.0
1,South Asia,132037.0
2,Sub-Saharan Africa,100557.0
3,South America,29388.0
4,Central America & Caribbean,28730.0
5,Southeast Asia,17109.0
6,Eastern Europe,7483.0
7,Western Europe,6709.0
8,North America,5153.0
9,East Asia,1154.0


In [59]:
# Attack outcomes
query = """
SELECT
    attack_success,
    COUNT(*) AS incidents
FROM terrorism
GROUP BY attack_success;
"""
pd.read_sql(query, conn)

,attack_success,incidents
0,Failed,24404
1,Successful,185302


In [60]:
# Suicide attack distribution
query = """
SELECT
    suicide_attack,
    COUNT(*) AS incidents
FROM terrorism
GROUP BY suicide_attack;
"""
pd.read_sql(query, conn)

,suicide_attack,incidents
0,No,202268
1,Yes,7438


In [61]:
# Historical trend by decade
query = """
SELECT
    decade,
    COUNT(*) AS incidents
FROM terrorism
GROUP BY decade
ORDER BY decade;
"""
pd.read_sql(query, conn)

,decade,incidents
0,1970s,9913
1,1980s,31156
2,1990s,28765
3,2000s,25057
4,2010s,106377
5,2020s,8438


In [66]:
# Events causing at least 100 casualties
query = """
SELECT
    attacktype1_txt,
    ROUND(AVG(casualties),2) AS avg_casualties
FROM terrorism
WHERE attacktype1_txt <> 'Unknown'
GROUP BY attacktype1_txt
ORDER BY avg_casualties DESC;
"""
pd.read_sql(query, conn)

,attacktype1_txt,avg_casualties
0,Hijacking,34.50
1,Unarmed Assault,12.81
2,Hostage Taking (Barricade Incident),8.29
3,Bombing/Explosion,5.87
4,Armed Assault,5.49
5,Hostage Taking (Kidnapping),2.94
6,Assassination,2.04
7,Facility/Infrastructure Attack,0.71


In [67]:
# Close database connection
conn.close()